In [1]:
# FMCW / NCO 验证（所有产物只写在 WC/ 目录）
import sys, time, numpy as np
sys.path.insert(0, '/home/xilinx/jupyter_notebooks/WC')
from fmcw_lab import Lab
lab = Lab()
lab.clock()
print(lab.lb.status())

  [PASS] xRFDC 已就绪                                 is_ready()=True
  [PASS] DAC0 进入 FULL                               state=0xF
  [PASS] ADC0 已使能                                  enabled=True
  [PASS] TX/RX DMA 已定位                             TX@0x80020000 RX@0x80000000
  读回窗口自检通过（3 字节传输，rx[2] = 数据字节）

=== 配置四线 SPI readback ===
  SPI_LOCK 解锁 (R0x1FFD/R0x1FFE=0x00, R0x1FFF=0x53)
  R0x000 = 0x10 (SPI_3WIRE_DIS=1)
  R0x15F 无条件写回 0x13（写前读到 0x13，脏状态下该读数不可信）
  R0x16E = 0x3B (PLL2_LD_MUX=7 readback, TYPE=3 推挽)
  读回可信度检查通过（R0x171=0xAA / R0x172=0x02）
→ PLL2 已锁定、SYSREF 连续输出，跳过写表
  LMK 现状: 读回可信=True  PLL1 LD=0(0x00)  PLL2 LD=1(0x02)  SYSREF MUX=3
  [PASS] LMK 读回可信                                 R0x171=0xAA / R0x172=0x02
  [PASS] LMK PLL2 锁定（RFDC 时钟源）                 R0x183=0x02
  [PASS] SYSREF 连续输出（MUX=3）                     R0x139 MUX=3
  PLL1 LD=0（本板 PLL1 空载，LD=0 正常，不是故障）

 2　时钟
  结果  项目                                         实测 / 说明
  --------------------------------------------------------

In [2]:
# 1) 单音：20 MHz
lab.program(20.0)
xc, r = lab.capture(16384)
f, amp = lab.tone(xc)
print('sr=0x%X std=%.0f 峰 %.3f MHz 幅度 %.0f' % (r['sr'], xc.std(), f/1e6, amp))

sr=0x5011 std=6517 峰 -20.000 MHz 幅度 13033


In [3]:
# 2) FMCW：20 -> 120 MHz, 100 us, 1 MHz/us
N = 25600; DSTEP = 4295000000
lab.program(20.0, ramp=1, chirp_len=N, d_step=DSTEP)
xc, r = lab.capture(65536)
print(lab.chirp_stats(xc))

{'period_us': 100.0, 'slope_mhz_us': -1.0000328412218886, 'f_start_mhz': -21.6706978249979, 'f_end_mhz': -118.32808540636616, 'rms_mhz': 0.3906876918823405, 'n_jumps': 3}


In [4]:
# 3) 幅度平坦度扫描
freqs = [5,15,25,35,45,55,65,75,85,95,105,115,121,124]
flat = lab.sweep_flatness(freqs)
for f, fm, a in flat: print('%6.1f -> %s MHz  幅度 %8.1f' % (f, 'None' if fm is None else round(fm,3), a))

   5.0 -> -5.0 MHz  幅度  12585.7
  15.0 -> -15.0 MHz  幅度  12987.9
  25.0 -> -25.0 MHz  幅度  13007.1
  35.0 -> -35.0 MHz  幅度  12712.8
  45.0 -> -45.0 MHz  幅度  12020.7
  55.0 -> -55.0 MHz  幅度  10961.0
  65.0 -> -65.0 MHz  幅度   9884.4
  75.0 -> -75.0 MHz  幅度   8792.5
  85.0 -> -85.0 MHz  幅度   7841.9
  95.0 -> -95.0 MHz  幅度   6999.6
 105.0 -> -105.0 MHz  幅度   6256.6
 115.0 -> -115.0 MHz  幅度   5612.9
 121.0 -> -121.0 MHz  幅度   5286.9
 124.0 -> -124.0 MHz  幅度   5143.2
